# A distance-covariate correlation cannot tell a graded field from a clustered one

The paper reports that cortical areas which are more strongly connected anatomically have
more similar geometry ($\rho = -0.51$), replicating Posani et al. (2026) with a shape
metric in place of averaged selectivity, and then asserts that this result is *equally
compatible with a graded and with a clustered organisation*.

Figure 1 does not establish that. It shows that **categoricality** and **pooled neuron
clustering** cannot distinguish the two. The connectivity result is a different statistic:
a correlation between a pairwise distance and an external pairwise covariate. Whether
*that* discriminates is a separate question, and this notebook answers it.

The three fields are figure 1's, unchanged:

| field | what it is |
|---|---|
| **gradient** | regions vary continuously; `a` and $\psi$ sweep together |
| **unstructured** | the same values, but the pairing between them is broken |
| **clustered** | no region is categorical; $\psi$ takes three discrete values, so the field genuinely has three region types |

To each we attach a connectivity matrix built the *same way*: strongly connected means close
in the underlying organisation, which is what connectivity means anatomically, plus
measurement noise. Then we ask both questions of each field: does distance track
connectivity, and does the region space contain types?

In [ ]:
from pathlib import Path

from shapemetrics import paths
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

R = paths.figure_code("Figure1").region_space
from shapemetrics import plotting

OUT = paths.set_figure("Figure1")
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

N_REGIONS, N_DRAWS = 30, 200
FIELDS = ("continuum", "no_gradient", "kinds")
NICE = {"continuum": "gradient", "no_gradient": "unstructured", "kinds": "clustered"}
IU = np.triu_indices(N_REGIONS, 1)

## The connectivity covariate

$\psi$ is what a region's geometry depends on, so proximity in $\psi$ is this simulation's
stand-in for proximity in the anatomical organisation. Log connectivity is set to
$-|\psi_i - \psi_j|$ plus Gaussian measurement noise, identically in all three fields, so
nothing about the *construction* of the covariate differs between them. Only the
organisation of the regions does.

In [ ]:
def psi_of(name, n=N_REGIONS, seed=0):
    """Each region's psi, matching R.scenario's own construction."""
    if name == "kinds":
        return np.tile(np.linspace(-R.PSI, R.PSI, R.N_KINDS), n // R.N_KINDS)
    t = np.sort(np.random.default_rng(seed).uniform(0, 1, n))
    return (-R.PSI + 2 * R.PSI * t) * R.PSI_SPAN


def log_connectivity(psi, noise, seed=1):
    """Strongly connected = close in the underlying organisation, plus noise."""
    rng = np.random.default_rng(seed)
    return (-np.abs(psi[IU[0]] - psi[IU[1]])
            + noise * np.std(psi) * rng.standard_normal(len(IU[0])))


f = OUT / "region_connectivity.npz"
if f.exists():
    d = np.load(f, allow_pickle=True)
    DM, CT = d["DM"].item(), d["CT"].item()
else:
    DM, CT = {}, {}
    for name in FIELDS:
        X, region, _ = R.scenario(name, n_regions=N_REGIONS)
        DM[name] = R.procrustes_distances(X, region)
        CT[name] = R.continuum_or_types(R.pca_embed(DM[name]), n_draws=N_DRAWS)
    np.savez(f, DM=np.array(DM, dtype=object), CT=np.array(CT, dtype=object))

for name in FIELDS:
    print(f"{NICE[name]:>13s}: region space  silhouette {CT[name]['obs']:.3f}"
          f"  null {CT[name]['null'].mean():.3f}"
          f"  z = {CT[name]['z']:+.2f}, p = {CT[name]['p']:.3f}")

## Both questions, on each field

At one fixed noise level, to start.

In [ ]:
NOISE0 = 0.2
print(f"{'field':>13s} {'rho(distance, log conn)':>25s} {'p':>10s}"
      f" {'region-space z':>16s} {'p':>8s}")
for name in FIELDS:
    lc = log_connectivity(psi_of(name), NOISE0)
    r = stats.spearmanr(DM[name][IU], lc)
    print(f"{NICE[name]:>13s} {r.statistic:+25.2f} {r.pvalue:10.1e}"
          f" {CT[name]['z']:+16.2f} {CT[name]['p']:8.3f}")
print("\nAll three: a strong, highly significant distance-connectivity correlation.")
print("Only one of them contains region types.")

## Does the *size* of the correlation discriminate?

The clustered field gives a larger $\rho$ above, so it is worth asking whether the
magnitude carries the information even if its significance does not.

It does not, because $\rho$ depends on how noisily connectivity is measured, which is a
property of the anatomical dataset rather than of the brain's organisation. Sweeping that
noise moves $\rho$ over a wide range in every field, and the ranges overlap almost
completely: a given observed value is reachable under either organisation.

In [ ]:
NOISES = np.array([0.0, 0.2, 0.5, 1.0, 2.0, 4.0])
RHO = {}
print(f"{'field':>13s} " + "".join(f"{f'noise {q:g}':>11s}" for q in NOISES))
for name in FIELDS:
    psi = psi_of(name)
    RHO[name] = np.array([stats.spearmanr(DM[name][IU],
                                          log_connectivity(psi, q)).statistic
                          for q in NOISES])
    print(f"{NICE[name]:>13s} " + "".join(f"{v:+11.2f}" for v in RHO[name]))

lo = max(min(RHO[n]) for n in FIELDS), min(max(RHO[n]) for n in FIELDS)
print(f"\nevery field can produce rho anywhere in [{lo[0]:+.2f}, {lo[1]:+.2f}]")
print("so the observed value alone identifies neither organisation")

## The same correlation, a different answer

The cleanest form of the claim: choose the noise in each field so that all three give
**the same** $\rho$, then ask the region-space question of each. The correlation is now
uninformative by construction, and the region-space test still separates them.

In [ ]:
TARGET = -0.60


def noise_for(name, target=TARGET):
    """The connectivity noise at which this field's rho hits the target."""
    grid = np.linspace(0, 5, 201)
    psi = psi_of(name)
    rho = np.array([stats.spearmanr(DM[name][IU],
                                    log_connectivity(psi, q)).statistic for q in grid])
    return grid[np.argmin(np.abs(rho - target))]


MATCH = {}
print(f"{'field':>13s} {'noise':>8s} {'rho':>8s} {'p':>10s}"
      f" {'region-space z':>16s} {'p':>8s}   verdict")
for name in FIELDS:
    q = noise_for(name)
    lc = log_connectivity(psi_of(name), q)
    r = stats.spearmanr(DM[name][IU], lc)
    MATCH[name] = dict(noise=q, rho=r.statistic, p=r.pvalue, conn=lc)
    v = "types" if CT[name]["p"] < 0.05 else "one continuous cloud"
    print(f"{NICE[name]:>13s} {q:8.2f} {r.statistic:+8.2f} {r.pvalue:10.1e}"
          f" {CT[name]['z']:+16.2f} {CT[name]['p']:8.3f}   {v}")

## The figure

In [ ]:
PANEL = 2.24
fig, axes = plt.subplots(2, 3, figsize=(3 * PANEL, 2.1 * PANEL))

for j, name in enumerate(FIELDS):
    ax = axes[0, j]
    m = MATCH[name]
    ax.scatter(m["conn"], DM[name][IU], s=5, color=plotting.GREY,
               edgecolor="none", rasterized=True)
    plotting.fit_line(ax, m["conn"], DM[name][IU])
    ax.set_title(NICE[name])
    ax.set_xlabel("log connectivity")
    if j == 0:
        ax.set_ylabel("Procrustes distance")
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.96, 0.94, f"$\\rho$ = {m['rho']:+.2f}", transform=ax.transAxes,
            ha="right", va="top", gid="point_label")

    ax = axes[1, j]
    ct = CT[name]
    plotting.null_hist(ax, ct["obs"], ct["null"], "best silhouette",
                       label_null="one blob", label_obs="regions", p=ct["p"],
                       strip_xticks=True)
    if j:
        ax.set_ylabel("")

for ax, letter in zip(axes.ravel(), "abcdef"):
    plotting.panel_letter(ax, letter, dx=-22)
plotting.typeset(fig)
fig.tight_layout(w_pad=1.0, h_pad=1.2)
plotting.save(fig, "connectivity_covariate", folder=str(OUT))

## What it shows

**Top row.** The three fields have been given connectivity measurements of differing
quality so that all three produce the same distance-connectivity correlation. By
construction the correlation now carries no information about which is which.

**Bottom row.** The region-space test separates them anyway. Only the clustered field
exceeds its null; the graded and unstructured fields are indistinguishable from a single
continuous cloud.

So the claim in the text holds, and this is what supports it: a significant correlation
between geometric distance and anatomical connectivity says the areas are *arranged*
with respect to connectivity, and says nothing about whether they fall into discrete
types. The second question needs the region space itself.

Two things worth being clear about. The covariate here is built from $\psi$, which is what
the geometry depends on, so this is the *best case* for the correlation: it is measuring
exactly the right latent, and it still cannot discriminate. And the correlations are
matched by varying measurement noise, which is a property of the connectivity dataset
rather than of the organisation, which is precisely why the observed value cannot be read
as evidence for either.